#### Phase 1: MISSING VALUES         → 1,100 rows with NaN in value
#### Phase 2: PLACEHOLDER VALUES     → 999.99 in PM2.5 (sensor error code)
#### Phase 3: NEGATIVE ANOMALIES     → 996 rows, all temperature = -52°C 
                                   (Gurugram is NOT Antarctica)
#### Phase 4: OUT-OF-RANGE OUTLIERS  → CO hits 49,300 µg/m³ (Chennai)
                                   PM10 hits 1,162
                                   SO2 hits 1,395
#### Phase 5: DTYPE FIX              → datetime column is a string, not datetime

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("../data/raw/india_aq_raw.csv")

In [6]:
df.shape

(105265, 8)

In [7]:
df.columns.tolist()

['location_id',
 'location',
 'sensor_id',
 'parameter',
 'units',
 'value',
 'datetime',
 'datetime_local']

In [8]:
df.dtypes

location_id         int64
location              str
sensor_id           int64
parameter             str
units                 str
value             float64
datetime              str
datetime_local        str
dtype: object

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 105265 entries, 0 to 105264
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   location_id     105265 non-null  int64  
 1   location        105265 non-null  str    
 2   sensor_id       105265 non-null  int64  
 3   parameter       105265 non-null  str    
 4   units           105265 non-null  str    
 5   value           104165 non-null  float64
 6   datetime        105265 non-null  str    
 7   datetime_local  105265 non-null  str    
dtypes: float64(1), int64(2), str(5)
memory usage: 13.7 MB


In [12]:
df.isna().sum()

location_id          0
location             0
sensor_id            0
parameter            0
units                0
value             1100
datetime             0
datetime_local       0
dtype: int64

In [13]:
df.head(10)

,location_id,location,sensor_id,parameter,units,value,datetime,datetime_local
0,17,R K Puram Delhi,12234783,no,ppb,93.2,2025-02-18T20:00:00Z,2025-02-19T01:30:00+05:30
1,17,R K Puram Delhi,12234783,no,ppb,90.4,2025-02-18T20:15:00Z,2025-02-19T01:45:00+05:30
2,17,R K Puram Delhi,12234783,no,ppb,84.7,2025-02-18T21:00:00Z,2025-02-19T02:30:00+05:30
3,17,R K Puram Delhi,12234783,no,ppb,80.8,2025-02-18T21:15:00Z,2025-02-19T02:45:00+05:30
4,17,R K Puram Delhi,12234783,no,ppb,81.6,2025-02-18T21:30:00Z,2025-02-19T03:00:00+05:30
5,17,R K Puram Delhi,12234783,no,ppb,94.8,2025-02-18T21:45:00Z,2025-02-19T03:15:00+05:30
6,17,R K Puram Delhi,12234783,no,ppb,100.2,2025-02-18T22:00:00Z,2025-02-19T03:30:00+05:30
7,17,R K Puram Delhi,12234783,no,ppb,97.8,2025-02-18T22:15:00Z,2025-02-19T03:45:00+05:30
8,17,R K Puram Delhi,12234783,no,ppb,71.5,2025-02-18T22:30:00Z,2025-02-19T04:00:00+05:30
9,17,R K Puram Delhi,12234783,no,ppb,62.6,2025-02-18T22:45:00Z,2025-02-19T04:15:00+05:30


In [14]:
df.describe()

,location_id,sensor_id,value
count,105265.000000,1.052650e+05,104165.000000
mean,216.067829,7.448437e+06,161.358028
std,146.540684,6.512123e+06,996.370450
min,17.000000,3.500000e+01,-52.030000
25%,50.000000,7.110000e+02,4.010000
50%,235.000000,1.223479e+07,31.500000
75%,354.000000,1.223561e+07,99.900000
max,407.000000,1.434179e+07,49300.000000


In [15]:
df[df['value'].isna()].groupby('parameter')['location'].count()

parameter
co                    2
no                    6
no2                   1
o3                   56
pm25                 31
relativehumidity    996
so2                   8
Name: location, dtype: int64

In [16]:
df[df['value'].isna()].groupby('parameter').size()

parameter
co                    2
no                    6
no2                   1
o3                   56
pm25                 31
relativehumidity    996
so2                   8
dtype: int64

In [18]:
df.groupby('location')['value'].mean()

location
Alandur Chennai         633.109744
Anand Vihar Delhi        92.516197
Collectorate Gaya       214.525415
Income Tax Delhi        234.611667
Lalbagh Mumbai          256.082022
Punjabi Bagh Delhi      135.261610
R K Puram Delhi         181.916159
Vikas Sadan Gurugram    131.372340
Zoo Park Hyderabad       98.396967
Name: value, dtype: float64

In [22]:
df.groupby('parameter')['value'].count()

parameter
co                  11227
no                   4994
no2                 13999
nox                  5000
o3                  12944
pm10                10036
pm25                13969
relativehumidity     4004
so2                 12992
temperature          5000
wind_direction       5000
wind_speed           5000
Name: value, dtype: int64

In [23]:
df.groupby(['location' , 'parameter'])['value'].max()

location            parameter       
Alandur Chennai     co                  49300.00
                    no2                    50.16
                    o3                    146.53
                    pm25                  999.99
                    so2                   405.06
                                          ...   
Zoo Park Hyderabad  relativehumidity       89.00
                    so2                    79.90
                    temperature            33.90
                    wind_direction        356.00
                    wind_speed              1.90
Name: value, Length: 78, dtype: float64

In [24]:
df.groupby('parameter')['units'].unique()

parameter
co                  [ppb, µg/m³]
no                         [ppb]
no2                 [ppb, µg/m³]
nox                        [ppb]
o3                       [µg/m³]
pm10                     [µg/m³]
pm25                     [µg/m³]
relativehumidity             [%]
so2                 [ppb, µg/m³]
temperature                  [c]
wind_direction             [deg]
wind_speed                 [m/s]
Name: units, dtype: object

In [25]:
df_clean = df.dropna(subset=['value'])

In [26]:
print(f"Before: {df.shape[0]} rows")
print(f"After:  {df_clean.shape[0]} rows")
print(f"Dropped: {df.shape[0] - df_clean.shape[0]} rows")

Before: 105265 rows
After:  104165 rows
Dropped: 1100 rows


In [32]:
null_counts = df[df["value"].isna()].groupby("parameter").size()
total_counts = df.groupby("parameter").size()
(null_counts / total_counts * 100).fillna(0).round(2)

parameter
co                   0.02
no                   0.12
no2                  0.01
nox                  0.00
o3                   0.43
pm10                 0.00
pm25                 0.22
relativehumidity    19.92
so2                  0.06
temperature          0.00
wind_direction       0.00
wind_speed           0.00
dtype: float64

In [33]:
df_clean[df_clean['value'] == 999.99]

,location_id,location,sensor_id,parameter,units,value,datetime,datetime_local
67261,301,Vikas Sadan Gurugram,1166,pm25,µg/m³,999.99,2016-03-28T00:45:00Z,2016-03-28T06:15:00+05:30
67262,301,Vikas Sadan Gurugram,1166,pm25,µg/m³,999.99,2016-03-28T01:00:00Z,2016-03-28T06:30:00+05:30
67263,301,Vikas Sadan Gurugram,1166,pm25,µg/m³,999.99,2016-03-28T01:15:00Z,2016-03-28T06:45:00+05:30
96099,378,Alandur Chennai,14304,pm25,µg/m³,999.99,2017-03-10T08:15:00Z,2017-03-10T13:45:00+05:30
96100,378,Alandur Chennai,14304,pm25,µg/m³,999.99,2017-03-10T08:45:00Z,2017-03-10T14:15:00+05:30


In [ ]:
for placeholder in[999, 999.99, -999, 9999]:
    count = len(df_clean[df_clean['value'] == placeholder])
    print(f"Placeholder {placeholder}: {count} rows")

Placeholder 999: 0 rows
Placeholder 999.99: 5 rows
Placeholder -999: 0 rows
Placeholder 9999: 0 rows


In [35]:
df_clean = df_clean[df_clean['value'] != 999.99]
print(f"After removing 999.99: {df_clean.shape[0]} rows")

After removing 999.99: 104160 rows


In [43]:
negatives = df_clean[df_clean['value'] < 0]
negatives.groupby(['location' , 'parameter']).size()

location              parameter  
Vikas Sadan Gurugram  temperature    996
dtype: int64

In [44]:
negatives[["location", "parameter", "value", "datetime"]].head(10)

,location,parameter,value,datetime
63126,Vikas Sadan Gurugram,temperature,-52.00,2025-10-01T06:30:00Z
63127,Vikas Sadan Gurugram,temperature,-52.00,2025-10-01T06:45:00Z
63128,Vikas Sadan Gurugram,temperature,-52.01,2025-10-01T07:00:00Z
63129,Vikas Sadan Gurugram,temperature,-52.01,2025-10-01T07:15:00Z
63130,Vikas Sadan Gurugram,temperature,-52.01,2025-10-01T07:30:00Z
63131,Vikas Sadan Gurugram,temperature,-52.01,2025-10-01T07:45:00Z
63132,Vikas Sadan Gurugram,temperature,-52.01,2025-10-01T08:00:00Z
63133,Vikas Sadan Gurugram,temperature,-52.01,2025-10-01T08:15:00Z
63134,Vikas Sadan Gurugram,temperature,-52.01,2025-10-01T08:30:00Z
63135,Vikas Sadan Gurugram,temperature,-52.01,2025-10-01T08:45:00Z


In [ ]:
bad_rows  = (
    (df_clean['location'] == 'Vikas Sadan Gurugram') &
    (df_clean['parameter'] == "temperature") &
    (df_clean['value'] < 0)
        
)

df_clean = df_clean[~bad_rows]


In [51]:
print(bad_rows.sum())
print(f"After removing bad rows: {df_clean.shape[0]} rows")

996
After removing bad rows: 103164 rows


In [52]:
thresholds = {
    "pm25": 500,
    "pm10": 600,
    "no2": 400,
    "o3": 300,
    "co": 10000,
    "so2": 200,
    "temperature": 50,
    "relativehumidity": 100,
    "wind_speed": 50,
}

for param , max_val in thresholds.items():
    count = len(df_clean[(df_clean['parameter'] == param) & (df_clean['value'] > max_val)])
    if count > 0:
        print(f"{count} rows with {param} > {max_val}")

174 rows with pm25 > 500
381 rows with pm10 > 600
10 rows with o3 > 300
82 rows with co > 10000
45 rows with so2 > 200


In [53]:
df_clean[df_clean['parameter'] == 'co'].nlargest(5, 'value')[['location' , 'value', 'datetime']]

,location,value,datetime
94209,Alandur Chennai,49300.0,2016-07-05T02:00:00Z
94188,Alandur Chennai,49140.0,2016-04-06T03:30:00Z
94254,Alandur Chennai,48500.0,2016-10-17T19:15:00Z
94246,Alandur Chennai,48400.0,2016-10-09T19:45:00Z
94200,Alandur Chennai,47550.0,2016-05-02T17:45:00Z


In [54]:
outlier_mask = pd.Series(False, index = df_clean.index)

for param, max_val in thresholds.items():
    bad = (df_clean["parameter"] == param) & (df_clean["value"] > max_val)
    outlier_mask = outlier_mask | bad
    
print(f"Total outliers flagged: {outlier_mask.sum()}")
df_clean = df_clean[~outlier_mask]
print(f"After removing outliers: {df_clean.shape[0]} rows")

Total outliers flagged: 692
After removing outliers: 102472 rows


In [57]:
df_clean['datetime'] = pd.to_datetime(df_clean['datetime'])
df_clean['datetime_local'] = pd.to_datetime(df_clean['datetime_local'])
print(df_clean.dtypes)


location_id                           int64
location                                str
sensor_id                             int64
parameter                               str
units                                   str
value                               float64
datetime                datetime64[us, UTC]
datetime_local    datetime64[us, UTC+05:30]
dtype: object


In [58]:
df_clean.to_csv("../data/processed/india_aq_clean.csv", index=False)
print(f"Saved {df_clean.shape[0]} clean rows!")

Saved 102472 clean rows!
